# RGBD Preprocessing — Single Sample Viewer

In [ ]:
# ── SET THESE TWO PATHS ───────────────────────────────────────────────────
IMAGE_PATH = "/absolute/path/to/sample.jpg"
BIN_PATH   = "/absolute/path/to/sample.bin"
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, importlib
import numpy as np
import cv2
import matplotlib.pyplot as plt

NOTEBOOK_DIR      = os.path.dirname(os.path.abspath("__file__"))
PREPROCESSING_DIR = os.path.join(NOTEBOOK_DIR, "preprocessing")
TUMOR_DATASET_DIR = os.path.join(PREPROCESSING_DIR, "TumorDataset")
for _p in [PREPROCESSING_DIR, TUMOR_DATASET_DIR]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import tumor_dataset; importlib.reload(tumor_dataset)
from tumor_dataset import Dataset
from preprocessing.preprocessing_functions._io import read_bin

In [ ]:
# Read the image
img_bgr = cv2.imread(IMAGE_PATH)

# Read the point cloud
grid_x, grid_y, grid_z = read_bin(BIN_PATH)

# Build the data_array tuple format the existing functions expect
data_arr = [(grid_x, grid_y, grid_z, os.path.basename(BIN_PATH))]

# Instantiate Dataset as a method host
ds = Dataset(data_path=os.path.dirname(IMAGE_PATH))

In [ ]:
# ── Run preprocessing steps ───────────────────────────────────────────────

# 1. Contour depth render (contourf → screenshot → grayscale)
depth_contour = ds.read_contours_array_depth(data_arr)[0]
depth_gray    = cv2.cvtColor(depth_contour, cv2.COLOR_BGR2GRAY) if depth_contour.ndim == 3 else depth_contour

# 2. RGD — blue channel replaced with depth
rgd = ds.infuse_depth_into_blue_channel([img_bgr], [depth_contour])[0]

# 3. RGBD-contour — RGB + depth as 4th channel
rgbd_contour = np.dstack([img_bgr.astype(np.float32), depth_gray.astype(np.float32)])

# 4. Raw grid_z — NaN fill + normalize (what read_contours_with_grid will produce)
grid_z_filled = np.nan_to_num(grid_z, nan=float(np.nanmin(grid_z)))
grid_z_norm   = (255 - cv2.normalize(grid_z_filled, None, 0, 255, cv2.NORM_MINMAX)).astype(np.uint8)

print("Done.")
print(f"  img_bgr       : {img_bgr.shape}")
print(f"  depth_gray    : {depth_gray.shape}")
print(f"  rgd           : {rgd.shape}")
print(f"  rgbd_contour  : {rgbd_contour.shape}")
print(f"  grid_z_norm   : {grid_z_norm.shape}")

In [ ]:
# ── Display ───────────────────────────────────────────────────────────────
panels = [
    (cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), "RGB",                   None),
    (depth_gray,                               "Depth (contourf)",      "gray"),
    (cv2.cvtColor(rgd, cv2.COLOR_BGR2RGB),     "RGD (blue replaced)",   None),
    (depth_gray,                               "RGBD-contour (4th ch)", "gray"),
    (grid_z_norm,                              "RGBD-rawgrid (4th ch)", "gray"),
]

fig, axes = plt.subplots(1, len(panels), figsize=(22, 4))
fig.suptitle(os.path.basename(IMAGE_PATH), fontsize=12)

for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()